In [1]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/model'
os.chdir(workdir)
os.getcwd()

'/vast/ai_projects/Papenfuss_SVEnsemble2/training'

In [2]:
import copy
import importlib

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr

import vcfparser as vcfp
import bndconvert as bndc

importlib.reload(vcfp)
importlib.reload(bndc)

<module 'bndconvert' from '/vast/ai_projects/Papenfuss_SVEnsemble2/training/bndconvert.py'>

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
# We need at least 1 path to population reference and 1 path to a callset
ref_path = 'data/gnomad.v4.1.sv.sites.vcf.gz'
callset_path = 'data/test_gridss_output.NA12878.vcf'

In [6]:
ref_df, ref_meta = vcfp.parse_vcf(ref_path, output_meta=True, line_limit=0, skip_format=True, info_limit_to=['END'])

Processed 0 entries                 Elapsed time: 0.0008454322814941406 seconds.
Processed 100000 entries                 Elapsed time: 6.772964000701904 seconds.
Processed 200000 entries                 Elapsed time: 13.596497058868408 seconds.
Processed 300000 entries                 Elapsed time: 20.481982231140137 seconds.
Processed 400000 entries                 Elapsed time: 27.347818851470947 seconds.
Processed 500000 entries                 Elapsed time: 34.21756362915039 seconds.
Processed 600000 entries                 Elapsed time: 40.97404885292053 seconds.
Processed 700000 entries                 Elapsed time: 47.850648403167725 seconds.
Processed 800000 entries                 Elapsed time: 54.61325669288635 seconds.
Processed 900000 entries                 Elapsed time: 61.299426794052124 seconds.
Processed 1000000 entries                 Elapsed time: 68.003093957901 seconds.
Processed 1100000 entries                 Elapsed time: 74.7940285205841 seconds.
Processed 120

In [7]:
ref_df = ref_df[ref_df.ALT == '<DEL>']

In [8]:
callset_df, callset_meta = vcfp.parse_vcf(callset_path, output_meta=True)

Processed 0 entries                 Elapsed time: 1.7404556274414062e-05 seconds.
Processed 100000 entries                 Elapsed time: 3.528468370437622 seconds.
Processed 200000 entries                 Elapsed time: 7.096893548965454 seconds.
Processed 300000 entries                 Elapsed time: 10.702305555343628 seconds.
Processed 400000 entries                 Elapsed time: 14.225421905517578 seconds.
Processed 500000 entries                 Elapsed time: 17.789915084838867 seconds.
Processed 600000 entries                 Elapsed time: 21.39964509010315 seconds.
Processed 700000 entries                 Elapsed time: 24.981608867645264 seconds.
Processed 800000 entries                 Elapsed time: 28.58347201347351 seconds.
Processed 900000 entries                 Elapsed time: 32.213820695877075 seconds.
Processed 1000000 entries                 Elapsed time: 35.75759792327881 seconds.
Finished processing. Processed 1007409 entries out of 1007410. Elapsed time: 54.598428010940

In [9]:
callset_df = bndc.bnd_to_event(callset_df)

In [10]:
def get_meta_missing_in_df(vcf_df, meta_dict):
    ids_for_removal = []
    for meta_id in vcfp.combine_meta_dict(meta_dict).keys():
        try:
            vcf_df[meta_id]
        except KeyError:
            ids_for_removal.append(meta_id)
    return ids_for_removal

def trim_meta(vcf_df, meta_dict):
    meta_local = vcfp.combine_meta_dict(meta_dict)
    missing_meta = get_meta_missing_in_df(vcf_df, meta_dict)
    print(f'Removing items {missing_meta} from meta since they are missing in the vcf_df')
    for item in missing_meta:
        del meta_local[item]
    meta_dict = vcfp.split_meta_dict(meta_local)
    return vcf_df, meta_dict

In [11]:
callset_df, callset_meta = trim_meta(callset_df, callset_meta)

Removing items ['FILTER_SINGLE_ASSEMBLY', 'INFO_ANRP', 'INFO_ANRPQ', 'INFO_ANSR', 'INFO_ANSRQ', 'INFO_AS', 'INFO_ASC', 'INFO_ASQ', 'INFO_BEALN', 'INFO_BEIDH', 'INFO_BEIDL', 'INFO_BMQ', 'INFO_BMQN', 'INFO_BMQX', 'INFO_BQ', 'INFO_BSC', 'INFO_BSCQ', 'INFO_BUM', 'INFO_BUMQ', 'INFO_BVF', 'INFO_CIPOS', 'INFO_CIRPOS', 'INFO_HOMSEQ', 'INFO_IHOMPOS', 'INFO_MATEID', 'INFO_RAS', 'INFO_RASQ', 'INFO_REF', 'INFO_REFPAIR', 'INFO_SB', 'INFO_SC', 'FORMAT_AF', 'FORMAT_ANRP', 'FORMAT_ANRPQ', 'FORMAT_ANSR', 'FORMAT_ANSRQ', 'FORMAT_ASQ', 'FORMAT_BQ', 'FORMAT_BSC', 'FORMAT_BSCQ', 'FORMAT_BUM', 'FORMAT_BUMQ', 'FORMAT_BVF', 'FORMAT_RASQ', 'FORMAT_REF', 'FORMAT_REFPAIR'] from meta since they are missing in the vcf_df


In [12]:
vcfp.delete_mult_val_cols(ref_df, ref_meta)
vcfp.convert_col_types(ref_df, ref_meta)

vcfp.delete_mult_val_cols(callset_df, callset_meta)
vcfp.convert_col_types(callset_df, callset_meta)

In [15]:
# ref_df_sample = ref_df.sample(n=50000).sort_values(by=['CHROM', 'INFO_END', 'POS'])
# ref_meta_sample = copy.deepcopy(ref_meta)
# callset_df_sample = callset_df.sample(n=10000).sort_values(by=['CHROM', 'POS', 'INFO_END'])
# callset_meta_sample = copy.deepcopy(callset_meta)

In [16]:
ref_df_range = ref_df.rename(columns={'CHROM': 'Chromosome', 'POS': 'Start', 'INFO_END': 'End'})
ref_df_range = pr.PyRanges(ref_df_range)

In [17]:
callset_df_range = callset_df.rename(columns={'CHROM': 'Chromosome', 'POS': 'Start', 'INFO_END': 'End'})
callset_df_range = pr.PyRanges(callset_df_range)

In [18]:
overlaps = ref_df_range.intersect(callset_df_range)

In [19]:
overlapping = callset_df_range.overlap(ref_df_range)

In [20]:
overlapped_df_range = callset_df_range.join(ref_df_range, how='left', report_overlap=True)

In [21]:
overlapped_df_range

,Chromosome,Start,ID,REF,LALT,QUAL,FORMAT_ASRP,FORMAT_ASSR,FORMAT_BAQ,FORMAT_BASRP,FORMAT_BASSR,FORMAT_CASQ,FORMAT_GT,FORMAT_IC,FORMAT_IQ,FORMAT_QUAL,FORMAT_RP,FORMAT_RPQ,FORMAT_SR,FORMAT_SRQ,FORMAT_VF,INFO_ASRP,INFO_ASSR,INFO_BA,INFO_BAQ,INFO_BASRP,INFO_BASSR,INFO_CAS,INFO_CASQ,INFO_CQ,INFO_EVENT,INFO_IC,INFO_INSRMP,INFO_INSRMRC,INFO_INSRMRO,INFO_INSRMRT,INFO_INSTAXID,INFO_IQ,INFO_MQ,INFO_MQN,INFO_MQX,INFO_RF,INFO_RP,INFO_RPQ,INFO_SR,INFO_SRQ,INFO_SVTYPE,INFO_VF,End,RALT,CHR2,ALT,Start_b,ID_b,REF_b,ALT_b,QUAL_b,End_b,length,Overlap
0,chr1,1845834,gridss0fb_1029,N,A[chr1:1845864[,2725.38,0,64,0.0,0,0,0.00,.,12,636.13,2725.38,1,14.92,6,112.22,45,0,64,0,0.0,0,0,0,0.00,2725.38,gridss0fb_1029,12,0.0,0,0,0,0,636.13,60.0,60.0,60.0,0,1,14.92,6,112.22,DEL,45,1845863,]chr1:1845833]C,chr1,<DEL>,1842397,gnomAD-SV_v3_DEL_chr1_afc5084d,N,<DEL>,999.0,1848340,5943,29
1,chr1,1845834,gridss0fb_1029,N,A[chr1:1845864[,2725.38,0,64,0.0,0,0,0.00,.,12,636.13,2725.38,1,14.92,6,112.22,45,0,64,0,0.0,0,0,0,0.00,2725.38,gridss0fb_1029,12,0.0,0,0,0,0,636.13,60.0,60.0,60.0,0,1,14.92,6,112.22,DEL,45,1845863,]chr1:1845833]C,chr1,<DEL>,1843883,gnomAD-SV_v3_DEL_chr1_3b7b2b3e,N,<DEL>,771.0,1865045,21162,29
2,chr1,1845834,gridss0fb_1029,N,A[chr1:1845864[,2725.38,0,64,0.0,0,0,0.00,.,12,636.13,2725.38,1,14.92,6,112.22,45,0,64,0,0.0,0,0,0,0.00,2725.38,gridss0fb_1029,12,0.0,0,0,0,0,636.13,60.0,60.0,60.0,0,1,14.92,6,112.22,DEL,45,1845863,]chr1:1845833]C,chr1,<DEL>,1845824,gnomAD-SV_v3_DEL_chr1_1b9bd6cb,N,<DEL>,725.0,1845902,78,29
3,chr1,2005917,gridss0fb_1143,N,C[chr1:2005937[,1697.97,0,44,0.0,0,0,0.00,.,10,422.84,1697.97,0,0.00,0,0.00,34,0,44,0,0.0,0,0,0,0.00,1697.97,gridss0fb_1143,10,0.0,0,0,0,0,422.84,60.0,60.0,60.0,0,0,0.00,0,0.00,DEL,34,2005936,]chr1:2005916]T,chr1,<DEL>,2001988,gnomAD-SV_v3_DEL_chr1_170f1bf4,N,<DEL>,87.0,2011988,10000,19
4,chr1,2005917,gridss0fb_1143,N,C[chr1:2005937[,1697.97,0,44,0.0,0,0,0.00,.,10,422.84,1697.97,0,0.00,0,0.00,34,0,44,0,0.0,0,0,0,0.00,1697.97,gridss0fb_1143,10,0.0,0,0,0,0,422.84,60.0,60.0,60.0,0,0,0.00,0,0.00,DEL,34,2005936,]chr1:2005916]T,chr1,<DEL>,2005622,gnomAD-SV_v3_DEL_chr1_d35dc4db,N,<DEL>,1.0,2006323,701,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5890904,chrX,120657544,gridss300fb_11,N,T[chrX:120657568[,154.74,7,0,0.0,0,0,154.74,.,0,0.00,154.74,0,0.00,0,0.00,7,7,0,0,0.0,0,0,1,154.74,154.74,gridss300fb_11,0,0.0,0,0,0,0,0.00,60.0,60.0,60.0,0,0,0.00,0,0.00,DEL,7,120657567,]chrX:120657543]A,chrX,<DEL>,-1,-1,-1,-1,-1.0,-1,-1,-120657545
5890905,chrX,120315479,gridss300fb_6,N,C[chrX:120315493[,103.00,5,0,0.0,0,0,103.00,.,0,0.00,103.00,0,0.00,0,0.00,5,5,0,0,0.0,0,0,1,103.00,103.00,gridss300fb_6,0,0.0,0,0,0,0,0.00,60.0,60.0,60.0,0,0,0.00,0,0.00,DEL,5,120315492,]chrX:120315478]A,chrX,<DEL>,-1,-1,-1,-1,-1.0,-1,-1,-120315480
5890906,chrX,129250333,gridss301fb_2,N,T[chrX:129250347[,180.00,9,0,0.0,0,0,180.00,.,0,0.00,180.00,0,0.00,0,0.00,9,9,0,0,0.0,0,0,1,180.00,180.00,gridss301fb_2,0,0.0,0,0,0,0,0.00,60.0,60.0,60.0,0,0,0.00,0,0.00,DEL,9,129250346,]chrX:129250332]A,chrX,<DEL>,-1,-1,-1,-1,-1.0,-1,-1,-129250334
5890907,chrX,129410656,gridss301fb_3,N,A[chrX:129410666[,243.62,11,0,0.0,0,0,243.62,.,0,0.00,243.62,0,0.00,0,0.00,11,11,0,0,0.0,0,0,1,243.62,243.62,gridss301fb_3,0,0.0,0,0,0,0,0.00,60.0,60.0,60.0,0,0,0.00,0,0.00,DEL,11,129410665,]chrX:129410655]A,chrX,<DEL>,-1,-1,-1,-1,-1.0,-1,-1,-129410657


In [23]:
overlapped_df_range = overlapped_df_range.as_df()

In [55]:
overlapped_df_range.loc[:, 'overlap_frac'] = \
    (overlapped_df_range['Overlap'] * 2 / \
    (- overlapped_df_range['Start'] + overlapped_df_range['End'] - \
    overlapped_df_range['Start_b'] + overlapped_df_range['End_b'])).clip(lower=0)
# overlapped_df_range.loc[:, 'overlap_callset_only'] = \
#     overlapped_df_range['Overlap'] / \
#     (- overlapped_df_range['Start'] + overlapped_df_range['End'])

/tmp/ipykernel_3178233/2955003197.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.54205607 0.05277778 0.00642628 ... 0.         0.         0.        ]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  overlapped_df_range.loc[:, 'overlap_frac'] = \


In [56]:
overlapped_df_range = overlapped_df_range.sort_values('overlap_frac', ascending=False).drop_duplicates(['ID'], keep='first').sort_index()

In [57]:
overlapped_df_range.loc[:, 'ref_matched'] = np.where(overlapped_df_range.overlap_frac >= 0.8, True, False)

In [62]:
overlapped_df_range.loc[:, 'length'] = overlapped_df_range['End'] - overlapped_df_range['Start']

In [64]:
overlapped_df_final = overlapped_df_range.drop(columns = [
    'Chromosome', 'Start', 'LALT', 'RALT', 'FORMAT_GT',
    'INFO_EVENT', 'INFO_SVTYPE', 'End', 'CHR2',
    'Start_b', 'ID_b', 'REF_b', 'ALT_b', 'End_b', 'QUAL_b', 'Overlap', 'overlap_frac'
])
overlapped_df_final = overlapped_df_final.set_index('ID')

In [65]:
overlapped_df_final

,REF,QUAL,FORMAT_ASRP,FORMAT_ASSR,FORMAT_BAQ,FORMAT_BASRP,FORMAT_BASSR,FORMAT_CASQ,FORMAT_IC,FORMAT_IQ,FORMAT_QUAL,FORMAT_RP,FORMAT_RPQ,FORMAT_SR,FORMAT_SRQ,FORMAT_VF,INFO_ASRP,INFO_ASSR,INFO_BA,INFO_BAQ,INFO_BASRP,INFO_BASSR,INFO_CAS,INFO_CASQ,INFO_CQ,INFO_IC,INFO_INSRMP,INFO_INSRMRC,INFO_INSRMRO,INFO_INSRMRT,INFO_INSTAXID,INFO_IQ,INFO_MQ,INFO_MQN,INFO_MQX,INFO_RF,INFO_RP,INFO_RPQ,INFO_SR,INFO_SRQ,INFO_VF,ALT,length,ref_matched
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
gridss0fb_1029,N,2725.38,0,64,0.0,0,0,0.00,12,636.13,2725.38,1,14.92,6,112.22,45,0,64,0,0.0,0,0,0,0.00,2725.38,12,0.0,0,0,0,0,636.13,60.00,60.0,60.0,0,1,14.92,6,112.22,45,<DEL>,29,False
gridss0fb_1143,N,1697.97,0,44,0.0,0,0,0.00,10,422.84,1697.97,0,0.00,0,0.00,34,0,44,0,0.0,0,0,0,0.00,1697.97,10,0.0,0,0,0,0,422.84,60.00,60.0,60.0,0,0,0.00,0,0.00,34,<DEL>,19,False
gridss0fb_1212,N,62.87,0,0,0.0,0,0,0.00,2,62.87,62.87,0,0.00,0,0.00,2,0,0,0,0.0,0,0,0,0.00,62.87,2,0.0,0,0,0,0,62.87,45.00,30.0,60.0,0,0,0.00,0,0.00,2,<DEL>,9,False
gridss0fb_1288,N,418.53,2,10,0.0,0,0,0.00,2,102.05,418.53,2,29.75,1,19.10,11,2,10,0,0.0,0,0,0,0.00,403.67,2,0.0,0,0,0,0,102.05,60.00,60.0,60.0,0,2,29.75,1,19.10,11,<DEL>,27,False
gridss0fb_133,N,197.05,0,0,0.0,0,0,0.00,0,0.00,197.05,13,197.05,0,0.00,13,0,0,0,0.0,0,0,0,0.00,197.05,0,0.0,0,0,0,0,0.00,42.38,20.0,60.0,0,13,197.05,0,0.00,13,<DEL>,223222762,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
gridss300fb_11,N,154.74,7,0,0.0,0,0,154.74,0,0.00,154.74,0,0.00,0,0.00,7,7,0,0,0.0,0,0,1,154.74,154.74,0,0.0,0,0,0,0,0.00,60.00,60.0,60.0,0,0,0.00,0,0.00,7,<DEL>,23,False
gridss300fb_6,N,103.00,5,0,0.0,0,0,103.00,0,0.00,103.00,0,0.00,0,0.00,5,5,0,0,0.0,0,0,1,103.00,103.00,0,0.0,0,0,0,0,0.00,60.00,60.0,60.0,0,0,0.00,0,0.00,5,<DEL>,13,False
gridss301fb_2,N,180.00,9,0,0.0,0,0,180.00,0,0.00,180.00,0,0.00,0,0.00,9,9,0,0,0.0,0,0,1,180.00,180.00,0,0.0,0,0,0,0,0.00,60.00,60.0,60.0,0,0,0.00,0,0.00,9,<DEL>,13,False


In [66]:
overlapped_df_final.to_csv('saved/gridss_overlaps.tsv', sep='\t')

In [45]:
ref_df_grouped, callset_df_grouped = ref_df.groupby('CHROM'), callset_df.groupby('CHROM')

In [46]:
callset_df_grouped.obj.loc[:, 'MATCH_FOUND'] = False
callset_df_grouped.obj.loc[:, 'MATCH_FRAC'] = 0

In [ ]:
for chrom, ref_chrom_df in ref_df_grouped:
    n_found = 0
    
    try:
        callset_chrom_df = callset_df_grouped.get_group(chrom)
    except KeyError:
        print(f'No CHROM {chrom} in callset, skipping')
        continue
    for ref_idx, ref_var in ref_chrom_df.iterrows():
        overlap_idx = []
        overlap_fracs = []
        for callset_idx, callset_var in callset_chrom_df.iterrows():
            # Once variants cannot overlap, stop iterating
            if callset_var['POS'] > ref_var['INFO_END']:
                break
            # If variants do not overlap, skip completely
            if callset_var['INFO_END'] < ref_var['POS']:
                continue
                
            overlap = min(callset_var['INFO_END'], ref_var['INFO_END']) - max(callset_var['POS'], ref_var['POS'])
            ref_len = ref_var['INFO_END'] - ref_var['POS']r
            callset_len = callset_var['INFO_END'] - callset_var['POS']
            overlap_frac = 2 * overlap / (ref_len + callset_len)
            
            if overlap_frac > 0.8:
                overlap_idx.append(callset_idx)
                overlap_fracs.append(overlap_frac)
            
        if len(overlap_idx) > 0:
            overlap_frac_max = 0
            for overlap_id, overlap_frac in zip(overlap_idx, overlap_fracs):
                if overlap_frac > overlap_frac_max:
                    overlap_id_max = overlap_id
                    overlap_frac_max = overlap_frac
            # print(f'Callset var {callset_df_grouped.obj.loc[overlap_id_max, :]} matched to ref var {ref_var}')
            callset_df_grouped.obj.loc[overlap_id_max, 'MATCH_FOUND'] = True
            callset_df_grouped.obj.loc[overlap_id_max, 'MATCH_FRAC'] = overlap_frac
            n_found += 1
        print(f'Found {n_found} matches', end='\r', flush=True)

In [43]:
callset_df_grouped.obj.loc[:, 'MATCH_FOUND'].sum()

26

In [116]:
ref_df_sample

,CHROM,POS,ID,REF,ALT,QUAL,INFO_CHR2,INFO_CPX_TYPE,INFO_END,INFO_END2,INFO_NCR,INFO_PCRMINUS_NCR,INFO_PCRPLUS_NCR,INFO_POS2,INFO_SOURCE,INFO_SVLEN,INFO_SVTYPE,INFO_UNRESOLVED_TYPE,INFO_AN,INFO_N_BI_GENOS,INFO_N_HOMREF,INFO_N_HET,INFO_N_HOMALT,INFO_FREQ_HOMREF,INFO_FREQ_HET,INFO_FREQ_HOMALT,INFO_CN_NUMBER,INFO_CN_NONREF_COUNT,INFO_CN_NONREF_FREQ,INFO_AN_afr,INFO_N_BI_GENOS_afr,INFO_N_HOMREF_afr,INFO_N_HET_afr,INFO_N_HOMALT_afr,INFO_FREQ_HOMREF_afr,INFO_FREQ_HET_afr,INFO_FREQ_HOMALT_afr,INFO_CN_NUMBER_afr,INFO_CN_NONREF_COUNT_afr,INFO_CN_NONREF_FREQ_afr,INFO_AN_ami,INFO_N_BI_GENOS_ami,INFO_N_HOMREF_ami,INFO_N_HET_ami,INFO_N_HOMALT_ami,INFO_FREQ_HOMREF_ami,INFO_FREQ_HET_ami,INFO_FREQ_HOMALT_ami,INFO_CN_NUMBER_ami,INFO_CN_NONREF_COUNT_ami,INFO_CN_NONREF_FREQ_ami,INFO_AN_amr,INFO_N_BI_GENOS_amr,INFO_N_HOMREF_amr,INFO_N_HET_amr,INFO_N_HOMALT_amr,INFO_FREQ_HOMREF_amr,INFO_FREQ_HET_amr,INFO_FREQ_HOMALT_amr,INFO_CN_NUMBER_amr,INFO_CN_NONREF_COUNT_amr,INFO_CN_NONREF_FREQ_amr,INFO_AN_asj,INFO_N_BI_GENOS_asj,INFO_N_HOMREF_asj,INFO_N_HET_asj,INFO_N_HOMALT_asj,INFO_FREQ_HOMREF_asj,INFO_FREQ_HET_asj,INFO_FREQ_HOMALT_asj,INFO_CN_NUMBER_asj,INFO_CN_NONREF_COUNT_asj,INFO_CN_NONREF_FREQ_asj,INFO_AN_eas,INFO_N_BI_GENOS_eas,INFO_N_HOMREF_eas,INFO_N_HET_eas,INFO_N_HOMALT_eas,INFO_FREQ_HOMREF_eas,INFO_FREQ_HET_eas,INFO_FREQ_HOMALT_eas,INFO_CN_NUMBER_eas,INFO_CN_NONREF_COUNT_eas,INFO_CN_NONREF_FREQ_eas,INFO_AN_fin,INFO_N_BI_GENOS_fin,INFO_N_HOMREF_fin,INFO_N_HET_fin,INFO_N_HOMALT_fin,INFO_FREQ_HOMREF_fin,INFO_FREQ_HET_fin,INFO_FREQ_HOMALT_fin,INFO_CN_NUMBER_fin,INFO_CN_NONREF_COUNT_fin,INFO_CN_NONREF_FREQ_fin,INFO_AN_mid,INFO_N_BI_GENOS_mid,INFO_N_HOMREF_mid,INFO_N_HET_mid,INFO_N_HOMALT_mid,INFO_FREQ_HOMREF_mid,INFO_FREQ_HET_mid,INFO_FREQ_HOMALT_mid,INFO_CN_NUMBER_mid,INFO_CN_NONREF_COUNT_mid,INFO_CN_NONREF_FREQ_mid,INFO_AN_nfe,INFO_N_BI_GENOS_nfe,INFO_N_HOMREF_nfe,INFO_N_HET_nfe,INFO_N_HOMALT_nfe,INFO_FREQ_HOMREF_nfe,INFO_FREQ_HET_nfe,INFO_FREQ_HOMALT_nfe,INFO_CN_NUMBER_nfe,INFO_CN_NONREF_COUNT_nfe,INFO_CN_NONREF_FREQ_nfe,INFO_AN_rmi,INFO_N_BI_GENOS_rmi,INFO_N_HOMREF_rmi,INFO_N_HET_rmi,INFO_N_HOMALT_rmi,INFO_FREQ_HOMREF_rmi,INFO_FREQ_HET_rmi,INFO_FREQ_HOMALT_rmi,INFO_CN_NUMBER_rmi,INFO_CN_NONREF_COUNT_rmi,INFO_CN_NONREF_FREQ_rmi,INFO_AN_sas,INFO_N_BI_GENOS_sas,INFO_N_HOMREF_sas,INFO_N_HET_sas,INFO_N_HOMALT_sas,INFO_FREQ_HOMREF_sas,INFO_FREQ_HET_sas,INFO_FREQ_HOMALT_sas,INFO_CN_NUMBER_sas,INFO_CN_NONREF_COUNT_sas,INFO_CN_NONREF_FREQ_sas,INFO_XX_NCR,INFO_XX_PCRMINUS_NCR,INFO_XX_PCRPLUS_NCR,INFO_XY_NCR,INFO_XY_PCRMINUS_NCR,INFO_XY_PCRPLUS_NCR,INFO_AN_XY,INFO_N_BI_GENOS_XY,INFO_N_HOMREF_XY,INFO_N_HET_XY,INFO_N_HOMALT_XY,INFO_FREQ_HOMREF_XY,INFO_FREQ_HET_XY,INFO_FREQ_HOMALT_XY,INFO_CN_NUMBER_XY,INFO_CN_NONREF_COUNT_XY,INFO_CN_NONREF_FREQ_XY,INFO_N_HEMIREF_XY,INFO_N_HEMIALT_XY,INFO_FREQ_HEMIREF_XY,INFO_FREQ_HEMIALT_XY,INFO_AN_XX,INFO_N_BI_GENOS_XX,INFO_N_HOMREF_XX,INFO_N_HET_XX,INFO_N_HOMALT_XX,INFO_FREQ_HOMREF_XX,INFO_FREQ_HET_XX,INFO_FREQ_HOMALT_XX,INFO_CN_NUMBER_XX,INFO_CN_NONREF_COUNT_XX,INFO_CN_NONREF_FREQ_XX,INFO_AN_afr_XY,INFO_N_BI_GENOS_afr_XY,INFO_N_HOMREF_afr_XY,INFO_N_HET_afr_XY,INFO_N_HOMALT_afr_XY,INFO_FREQ_HOMREF_afr_XY,INFO_FREQ_HET_afr_XY,INFO_FREQ_HOMALT_afr_XY,INFO_CN_NUMBER_afr_XY,INFO_CN_NONREF_COUNT_afr_XY,INFO_CN_NONREF_FREQ_afr_XY,INFO_N_HEMIREF_afr_XY,INFO_N_HEMIALT_afr_XY,INFO_FREQ_HEMIREF_afr_XY,INFO_FREQ_HEMIALT_afr_XY,INFO_AN_afr_XX,INFO_N_BI_GENOS_afr_XX,INFO_N_HOMREF_afr_XX,INFO_N_HET_afr_XX,INFO_N_HOMALT_afr_XX,INFO_FREQ_HOMREF_afr_XX,INFO_FREQ_HET_afr_XX,INFO_FREQ_HOMALT_afr_XX,INFO_CN_NUMBER_afr_XX,INFO_CN_NONREF_COUNT_afr_XX,INFO_CN_NONREF_FREQ_afr_XX,INFO_AN_ami_XY,INFO_N_BI_GENOS_ami_XY,INFO_N_HOMREF_ami_XY,INFO_N_HET_ami_XY,INFO_N_HOMALT_ami_XY,INFO_FREQ_HOMREF_ami_XY,INFO_FREQ_HET_ami_XY,INFO_FREQ_HOMALT_ami_XY,INFO_CN_NUMBER_ami_XY,INFO_CN_NONREF_COUNT_ami_XY,INFO_CN_NONREF_FREQ_ami_XY,INFO_N_HEMIREF_ami_XY,INFO_N_HEMIALT_ami_XY,INFO_FREQ_HEMIREF_ami_XY,INFO_FREQ_HEMIALT_ami_XY,INFO_AN_ami_XX,INFO_N_BI_GENOS_ami_XX,I

In [117]:
callset_df_sample

,CHROM,POS,ID,REF,LALT,QUAL,FORMAT_ASRP,FORMAT_ASSR,FORMAT_BAQ,FORMAT_BASRP,FORMAT_BASSR,FORMAT_CASQ,FORMAT_GT,FORMAT_IC,FORMAT_IQ,FORMAT_QUAL,FORMAT_RP,FORMAT_RPQ,FORMAT_SR,FORMAT_SRQ,FORMAT_VF,INFO_ASRP,INFO_ASSR,INFO_BA,INFO_BAQ,INFO_BASRP,INFO_BASSR,INFO_CAS,INFO_CASQ,INFO_CQ,INFO_EVENT,INFO_IC,INFO_INSRMP,INFO_INSRMRC,INFO_INSRMRO,INFO_INSRMRT,INFO_INSTAXID,INFO_IQ,INFO_MQ,INFO_MQN,INFO_MQX,INFO_RF,INFO_RP,INFO_RPQ,INFO_SR,INFO_SRQ,INFO_SVTYPE,INFO_VF,INFO_END,RALT,CHR2,ALT
520,chr1,99070,gridss0fb_53,N,C[chr1:99082[,141.96,0,0,0.0,0,0,0.00,.,4,141.96,141.96,0,0.00,0,0.0,4,0,0,0,0.0,0,0,0,0.00,141.96,gridss0fb_53,4,0.0,0,0,0,0,141.96,42.75,39.0,47.0,0,0,0.00,0,0.0,DEL,4,99081,]chr1:99069]T,chr1,<DEL>
698,chr1,602271,gridss0fb_99,N,T[chr1:602283[,70.98,0,0,0.0,0,0,0.00,.,2,70.98,70.98,0,0.00,0,0.0,2,0,0,0,0.0,0,0,0,0.00,70.98,gridss0fb_99,2,0.0,0,0,0,0,70.98,48.50,41.0,56.0,0,0,0.00,0,0.0,DEL,2,602282,]chr1:602270]C,chr1,<DEL>
402,chr1,876293,gridss0fb_363,N,C[chr1:876303[,65.75,0,0,0.0,0,0,0.00,.,2,65.75,65.75,0,0.00,0,0.0,2,0,0,0,0.0,0,0,0,0.00,65.75,gridss0fb_363,2,0.0,0,0,0,0,65.75,60.00,60.0,60.0,0,0,0.00,0,0.0,DEL,2,876302,]chr1:876292]C,chr1,<DEL>
409,chr1,882949,gridss0fb_368,N,T[chr1:882963[,205.44,0,0,0.0,0,0,0.00,.,5,175.69,205.44,2,29.75,0,0.0,7,0,0,0,0.0,0,0,0,0.00,174.70,gridss0fb_368,5,0.0,0,0,0,0,175.69,40.86,33.0,60.0,0,2,29.75,0,0.0,DEL,7,882962,]chr1:882948]T,chr1,<DEL>
494,chr1,1048773,gridss0fb_496,N,A[chr1:1048784[,715.52,0,17,0.0,0,0,0.00,.,6,209.97,715.52,0,0.00,0,0.0,11,0,17,0,0.0,0,0,0,0.00,680.52,gridss0fb_496,6,0.0,0,0,0,0,209.97,60.00,60.0,60.0,0,0,0.00,0,0.0,DEL,11,1048783,]chr1:1048772]A,chr1,<DEL>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144510,chrX,135563651,gridss301fb_11,N,T[chrX:135563677[,221.48,10,0,0.0,0,0,221.48,.,0,0.00,221.48,0,0.00,0,0.0,10,10,0,0,0.0,0,0,1,221.48,221.48,gridss301fb_11,0,0.0,0,0,0,0,0.00,60.00,60.0,60.0,0,0,0.00,0,0.0,DEL,10,135563676,]chrX:135563650]A,chrX,<DEL>
144511,chrX,136023171,gridss301fb_12,N,T[chrX:136023217[,132.89,6,0,0.0,0,0,132.89,.,0,0.00,132.89,0,0.00,0,0.0,6,6,0,0,0.0,0,0,1,132.89,132.89,gridss301fb_12,0,0.0,0,0,0,0,0.00,60.00,60.0,60.0,0,0,0.00,0,0.0,DEL,6,136023216,]chrX:136023170]A,chrX,<DEL>
144533,chrX,139922233,gridss302fb_3,N,A[chrX:139922261[,110.74,5,0,0.0,0,0,110.74,.,0,0.00,110.74,0,0.00,0,0.0,5,5,0,0,0.0,0,0,1,110.74,110.74,gridss302fb_3,0,0.0,0,0,0,0,0.00,60.00,60.0,60.0,0,0,0.00,0,0.0,DEL,5,139922260,]chrX:139922232]T,chrX,<DEL>
144526,chrX,142155527,gridss302fb_15,N,T[chrX:146980255[,153.74,7,0,0.0,0,0,153.74,.,0,0.00,153.74,0,0.00,0,0.0,7,7,0,0,0.0,0,0,1,153.74,153.74,gridss302fb_15,0,0.0,0,0,0,0,0.00,60.00,60.0,60.0,0,0,0.00,0,0.0,DEL,7,146980254,]chrX:142155526]A,chrX,<DEL>


In [10]:
def combine_meta_dict(meta_dict):
    """
    Accepts a nested meta_dict to unwrap it and return a meta dict where all the fields are mixed
    """
    return {**meta_dict['FILTER'], **meta_dict['INFO'], **meta_dict['FORMAT']}

def split_meta_dict(meta_dict_combined):
    """
    Accepts an unwrapped meta_dict to return it to nested form
    """
    meta_dict = {
        'FILTER': {},
        'INFO': {},
        'FORMAT': {}
    }
    for meta_key, meta_item in meta_dict_combined.items():
        meta_dict[re.match(r'(INFO)|(FORMAT)|(FILTER)', meta_key).group()][meta_key] = meta_item
    
    return meta_dict

def get_split_meta_entry(meta_id, meta_info, values, output_type='Flag'):
    """
    If a meta entry has multiple values, fetches them all and turns them into separate meta entries 
    with name: {meta_id}_{value}

    Returns a dictionary with new ids as keys and tuples of (1, specified output type name, description)
    """
    new_ids = [f'{meta_id}_{value}' for value in values]

    return {new_id: (1, output_type, f'SPLIT FROM {meta_id}, OLD DESC: {meta_info[2]}') for new_id in new_ids}
    

def get_processed_algorithm_df_and_meta(vcf_df, meta_dict, delim=',', alg_col_name='INFO_ALGORITHMS'):
    """
    
    """
    # algs = vcf_df[alg_col].str.split(delim).astype(set)
    alg_col = vcf_df[alg_col_name]
    alg_set = set()
    
    for uniq_alg_comb in alg_col.unique():
        alg_set = alg_set | set(uniq_alg_comb.split(delim))
        
    alg_dict = {f'{alg_col_name}_{key}': [0] * len(alg_col) for key in alg_set}
    
    for i, alg_item in enumerate(alg_col):
        for alg in alg_item.split(delim):
            alg_dict[f'{alg_col_name}_{alg}'][i] = 1

    meta_dict_copy = copy.deepcopy(meta_dict)
    old_alg_entry = meta_dict_copy['INFO'].pop(alg_col_name)
    new_alg_entry = get_split_meta_entry(alg_col_name, old_alg_entry, alg_set)
    # print(old_alg_entry, new_alg_entry)

    # print(pd.concat([vcf_df[vcf_df.columns.drop([alg_col_name])], pd.DataFrame(alg_dict)], axis=1))
    meta_dict_copy['INFO'].update(new_alg_entry)
    
    return pd.concat([vcf_df[vcf_df.columns.drop([alg_col_name])], pd.DataFrame(alg_dict)], axis=1), meta_dict_copy
        

def convert_col_types(vcf_df, meta_dict, types_from='meta'):
    """
    Reads the types for INFO and FORMAT fields from the meta_dict and uses them to convert fields in vcf_df into the specified types
    """
    if types_from == 'meta':
        type_convert_dict = {
            'String': str,
            'Integer': int,
            'Float': float,
            'Flag': int
        }
        for meta_id, meta_info in (meta_dict['INFO'] | meta_dict['FORMAT']).items():
            vcf_df[meta_id] = vcf_df[meta_id].astype(type_convert_dict[meta_info[1]])


def delete_mult_val_cols(vcf_df, meta_dict, exceptions='default', vocal=False):
    """
    
    """
    if exceptions == 'default':
        exceptions = ['INFO_ALGORITHMS']
    if exceptions is None:
        exceptions = []
    
    keys_for_deletion = []
    for meta_group in meta_dict.values():
        for meta_id, meta_info in meta_group.items():
            # print(meta_id, meta_info)
            if meta_info[0] not in ['1', 1] + exceptions:
                keys_for_deletion.append(meta_id)

    if vocal:
        print(f'Dropping entries {keys_for_deletion} since they contain multiple values')

    for key_for_deletion in keys_for_deletion:
        del meta_dict[re.match(r'(INFO)|(FORMAT)|(FILTER)', key_for_deletion).group()][key_for_deletion]
    
    vcf_df.drop(keys_for_deletion, axis=1, inplace=True)

In [ ]:
def split_vcf_by_chrom(vcf_df, chrom_list=None):
    if chrom_list is not None:
        return {chrom: vcf_df.loc[vcf_df.CHROM == chrom] \
                for chrom in chrom_list}
    return {chrom: gnomad_vcf_df.loc[gnomad_vcf_df.CHROM == chrom] \
                for chrom in pd.unique(vcf_df.CHROM)}


def generate_truth_list(vcf_df, ref_df_dict, pos_interval=25, report_freq=100000):

    truth_list = []
    pos_series = vcf_df.POS
    chrom_series = vcf_df.CHROM
    
    if set(ref_df_dict.keys()) != set(chrom_series.unique()):
        raise KeyError('Chromosomes in prediction dataset do not match chromosomes in gnomad')
    
    for i, (pos, chrom) in enumerate(zip(pos_series, chrom_series)):
        # print(f'{chrom}: {pos}')
        # overlapping_ref_positions = (gnomad_df[chrom].POS > pos - 25) & (gnomad_df[chrom].POS < pos + 25)
    
        truth_list.append(
            ((ref_df_dict[chrom].POS > pos - pos_interval) & (ref_df_dict[chrom].POS < pos + pos_interval)).any()
        )
        
        if i % report_freq == 0:
            print(f'processed {i} entries')

    return truth_list


# Global vars
wd = '/stornext/Bioinf/data/bioinf-data/Papenfuss_lab_projects/SVensemble2/vcf_parsing'
gridss_path = f'{wd}/data/test_gridss_output.NA12878.vcf'
gnomad_path = '/home/users/allstaff/shikov.v/projects/SVensemble2/data/gnomad/4.1.0/downloads/gnomad.v4.1.sv.sites.vcf.gz'

gnomad_vcf_df, gnomad_meta = VCF_parser.parse_vcf(gnomad_path, output_meta=True)
gridss_vcf_df, gridss_meta = VCF_parser.parse_vcf(gridss_path, output_meta=True)

# gnomad_df_copy = gnomad_df.copy(deep=True)
# gnomad_meta_copy = copy.deepcopy(gnomad_meta)
gnomad_vcf_df, gnomad_meta = VCF_parser.get_processed_algorithm_df_and_meta(gnomad_vcf_df, gnomad_meta)
VCF_parser.delete_mult_val_cols(gnomad_vcf_df, gnomad_meta)
VCF_parser.convert_col_types(gnomad_vcf_df, gnomad_meta)

VCF_parser.delete_mult_val_cols(gridss_vcf_df, gridss_meta)
VCF_parser.convert_col_types(gridss_vcf_df, gridss_meta)

gnomad_vcf_df = split_vcf_by_chrom(gnomad_vcf_df, pd.unique(gridss_vcf_df.CHROM))

truth_list = generate_truth_list(gridss_vcf_df, gnomad_vcf_df)
gridss_vcf_df['MATCH_FOUND'] = truth_list

gnomad_vcf_df = pd.concat(gnomad_vcf_df, ignore_index=True)

gnomad_vcf_df = gnomad_vcf_df.sort_values(by=['POS'])
gridss_vcf_df = gridss_vcf_df.sort_values(by=['POS'])

truth_suffix = '_truth_set'
gnomad_vcf_df = gnomad_vcf_df.add_suffix(truth_suffix)

gridss_vcf_df = pd.concat(
    [
        gridss_vcf_df.loc[~gridss_vcf_df['MATCH_FOUND']],
        pd.merge_asof(
            gridss_vcf_df.loc[gridss_vcf_df['MATCH_FOUND']],
            gnomad_vcf_df,
            left_on='POS', right_on=f'POS{truth_suffix}',
            left_by='CHROM', right_by=f'CHROM{truth_suffix}',
            direction='nearest'
        )
    ]
)

cols_to_drop = [f'CHROM{truth_suffix}', f'POS{truth_suffix}', f'REF{truth_suffix}']
gridss_vcf_df = gridss_vcf_df[gridss_vcf_df.columns.drop(cols_to_drop)]

gridss_vcf_df = pd.concat(
    [
        gridss_vcf_df[['CHROM', 'ID', 'REF']],
        gridss_vcf_df.select_dtypes(include=np.number),
        gridss_vcf_df[['MATCH_FOUND']],
        gridss_vcf_df.filter(regex=f'INFO_ALGORITHMS{truth_suffix}')
    ],
    axis=1
)

gridss_vcf_df.to_csv('~/projects/SVensemble2/truth_set_gen/saved/gridss_vcf_df.filtered.tsv', sep='\t')
